# MotionStitch - Pose Tracking for Indian classical dance forms

In [8]:
# mediapipe: extracts body joint positions from video frames
# opencv-python: reads the video file frame by frame
# yt-dlp: downloads a video from a URL (e.g. a credited public tutorial) without saving it permanently
!pip install -U yt-dlp -q
!pip install mediapipe opencv-python -q

## Download the pose-tracking model

In [9]:
import urllib.request

# This is Google's official pretrained body-pose model — it's what actually
# figures out where each joint is in every frame. We're not training this
# ourselves; we're using it as-is to process our reference videos.
MODEL_URL = "https://storage.googleapis.com/mediapipe-models/pose_landmarker/pose_landmarker_lite/float16/1/pose_landmarker_lite.task"
urllib.request.urlretrieve(MODEL_URL, "pose_landmarker.task")
print("Model downloaded.")

Model downloaded.


## List your source videos

In [10]:
VIDEOS_TO_PROCESS = [
    {
        "adavu_name": "natta_adavu_1_8_v1",
        "source": "https://www.youtube.com/watch?v=5ILFJnEATlM",
        "credit": "YouTube creator, Lesson 12 — Natta Adavu 1-8 (PENDING PERMISSION before any public release)",
    },
    {
        "adavu_name": "natta_adavu_1_8_v2",
        "source": "https://www.youtube.com/watch?v=_rged7LZk4Y",
        "credit": "YouTube creator, Lesson 2 — Nattu Adavu 1-8 (PENDING PERMISSION before any public release)",
    },
    {
        "adavu_name": "natta_adavu_speeds",
        "source": "https://www.youtube.com/watch?v=OIKOHzePJCA",
        "credit": "YouTube creator, Basics Ep. 16 — Nattadavu 1-4, three speeds (PENDING PERMISSION before any public release)",
    },
]

## 4. Download a video temp

In [11]:
import subprocess

def download_video(url: str, out_path: str = "temp_video.mp4") -> str:
    """Downloads a video to a temporary local file in THIS Colab session.
    Runs the yt-dlp command-line tool directly (rather than its Python API)
    so we can reliably pass --remote-components, which lets yt-dlp fetch
    the small JS challenge-solving script YouTube now requires."""
    subprocess.run([
        "yt-dlp",
        "--remote-components", "ejs:github",
        "--cookies", "cookies.txt",
        "-f", "mp4/best",
        "-o", out_path,
        url,
    ], check=True)
    return out_path

## 5. The actual pose extraction

In [12]:
import cv2
import mediapipe as mp
import numpy as np
import json

BaseOptions = mp.tasks.BaseOptions
PoseLandmarker = mp.tasks.vision.PoseLandmarker
PoseLandmarkerOptions = mp.tasks.vision.PoseLandmarkerOptions
VisionRunningMode = mp.tasks.vision.RunningMode

def compute_motion_magnitude(pose_sequence: list) -> np.ndarray:
    """
    For each frame, measures total body movement vs. the previous frame,
    summed across all 33 joints. Talking = low, sustained values.
    Dancing = high, sustained values. This is how we find the dance
    automatically, without watching the video ourselves.
    """
    magnitudes = [0.0]
    for i in range(1, len(pose_sequence)):
        prev_joints = pose_sequence[i - 1]["joints"]
        curr_joints = pose_sequence[i]["joints"]
        if prev_joints is None or curr_joints is None:
            magnitudes.append(0.0)
            continue
        total = sum(
            np.sqrt((c["x"] - p["x"]) ** 2 + (c["y"] - p["y"]) ** 2 + (c["z"] - p["z"]) ** 2)
            for p, c in zip(prev_joints, curr_joints)
        )
        magnitudes.append(total)
    return np.array(magnitudes)


def smooth(values: np.ndarray, window: int = 15) -> np.ndarray:
    """Rolling average — a single quick gesture while talking shouldn't
    count as 'dancing'; we care about SUSTAINED motion."""
    kernel = np.ones(window) / window
    return np.convolve(values, kernel, mode="same")


def auto_trim_to_active_segment(pose_sequence: list, threshold_percentile: float = 60) -> list:
    """
    Finds the single longest continuous stretch of high-motion frames
    (the real dancing) and discards everything else — intro talking,
    standing still, walking into position — automatically.
    """
    smoothed = smooth(compute_motion_magnitude(pose_sequence))
    threshold = np.percentile(smoothed, threshold_percentile)
    is_active = smoothed > threshold

    best_start, best_end, best_length = 0, 0, 0
    current_start = None
    for i, active in enumerate(is_active):
        if active and current_start is None:
            current_start = i
        elif not active and current_start is not None:
            if i - current_start > best_length:
                best_start, best_end, best_length = current_start, i, i - current_start
            current_start = None
    if current_start is not None and len(is_active) - current_start > best_length:
        best_start, best_end = current_start, len(is_active)

    return pose_sequence[best_start:best_end]

def extract_pose_sequence(video_path: str, frame_skip: int = 1) -> list:
    """
    Runs pose detection on the video. frame_skip=1 processes every frame;
    frame_skip=3 processes 1 out of every 3 (faster, still smooth enough
    for dance movement). Uses the video's own reported timestamp for each
    frame rather than a hand-calculated one, so timestamps stay correctly
    increasing no matter which frames get skipped.
    """
    options = PoseLandmarkerOptions(
        base_options=BaseOptions(model_asset_path="pose_landmarker.task"),
        running_mode=VisionRunningMode.VIDEO,
    )

    cap = cv2.VideoCapture(video_path)
    sequence = []
    frame_count = 0
    last_timestamp_ms = -1

    with PoseLandmarker.create_from_options(options) as landmarker:
        while cap.isOpened():
            success, frame = cap.read()
            if not success:
                break

            frame_count += 1
            if frame_count % frame_skip != 0:
                continue  # skip detection on this frame, but keep reading forward

            # the video's own reported position — always correctly
            # increasing over time, regardless of which frames we skip
            timestamp_ms = int(cap.get(cv2.CAP_PROP_POS_MSEC))

            # safety net: some video files occasionally report a duplicate
            # or non-increasing timestamp — this guarantees it always advances
            if timestamp_ms <= last_timestamp_ms:
                timestamp_ms = last_timestamp_ms + 1
            last_timestamp_ms = timestamp_ms

            rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_frame)

            result = landmarker.detect_for_video(mp_image, timestamp_ms)

            if result.pose_landmarks:
                joints = [{"x": lm.x, "y": lm.y, "z": lm.z} for lm in result.pose_landmarks[0]]
            else:
                joints = None

            sequence.append({"t_ms": timestamp_ms, "joints": joints})

    cap.release()
    return sequence

## 6. Run it on your whole list, save results, clean up

In [13]:
!curl -fsSL https://deno.land/install.sh | sh -s -- -y --no-modify-path
import os
os.environ["PATH"] += ":/root/.deno/bin"

######################################################################## 100.0%
Archive:  /root/.deno/bin/deno.zip
  inflating: /root/.deno/bin/deno    
Deno was installed successfully to /root/.deno/bin/deno
Download https://jsr.io/@deno/installer-shell-setup/meta.json
Download https://jsr.io/@deno/installer-shell-setup/0.3.4_meta.json
Run 'deno --help' to get started

Stuck? Join our Discord https://discord.gg/deno


In [14]:
import os

os.makedirs("clip_library", exist_ok=True)
manifest = []

for entry in VIDEOS_TO_PROCESS:
    print(f"Processing: {entry['adavu_name']}")

    video_path = download_video(entry["source"])
    full_sequence = extract_pose_sequence(video_path, frame_skip=3)
    pose_sequence = auto_trim_to_active_segment(full_sequence)
    print(f"  Auto-trimmed {len(full_sequence)} frames → {len(pose_sequence)} frames")

    output_path = f"clip_library/{entry['adavu_name']}.json"
    with open(output_path, "w") as f:
        json.dump({
            "adavu_name": entry["adavu_name"],
            "credit": entry["credit"],
            "frame_count": len(pose_sequence),
            "frames": pose_sequence,
        }, f)

    manifest.append({"adavu_name": entry["adavu_name"], "file": f"{entry['adavu_name']}.json", "credit": entry["credit"]})

    os.remove(video_path)  # delete the video — only the small JSON survives
    print(f"  Saved {output_path}, video deleted.")

with open("clip_library/manifest.json", "w") as f:
    json.dump(manifest, f, indent=2)

print("Done. Only clip_library/ (small JSON files) remains.")

Processing: natta_adavu_1_8_v1
  Auto-trimmed 3266 frames → 137 frames
  Saved clip_library/natta_adavu_1_8_v1.json, video deleted.
Processing: natta_adavu_1_8_v2
  Auto-trimmed 14959 frames → 176 frames
  Saved clip_library/natta_adavu_1_8_v2.json, video deleted.
Processing: natta_adavu_speeds
  Auto-trimmed 2512 frames → 52 frames
  Saved clip_library/natta_adavu_speeds.json, video deleted.
Done. Only clip_library/ (small JSON files) remains.


## 7. Download the results to your actual computer

In [15]:
from google.colab import files
import shutil

shutil.make_archive("clip_library", "zip", "clip_library")
files.download("clip_library.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>